# Week 0 Try It Yourself - Solution-Diffusion Pyomo Model

This try-it-yourself (TIY) exercise is intended to get you familiar with creating Pyomo model components that are commonly used in WaterTAP flowsheets: Var, Param, Expression, Constraint, and Objective.

In this exercise, you will create a Pyomo implementation of the solution-diffusion (SD) model, which describes the transport of water and salt through membranes.

We have provided a starting point that includes adding parameters needed, the concentration ($C_i$) and flow ($Q_i$) variables, and calculation of the osmotic pressure ($\pi_i$).

You will need to add each of the following variables as a `Var` to the model:

- system recovery $R$
- water permeability coefficient $K_w$ (LMH/bar)
- salt permeability coefficient $K_s$ (LMH)
- membrane area $A$ (m2)
- pump pressure $\Delta P$ (bar)
- pump power $W_p$ (kW)
- water flux $J_w$ (LMH)
- salt flux $J_s$ (g/(m2 hr))


These variables will be defined by the following equations. You will need to add each as a `Constraint` to the model:

_Mass Balance_
  
  $$
  Q_f C_f = Q_p C_p + Q_b C_b
  $$

_Flow Balance_
  
  $$
  Q_f = Q_p + Q_b
  $$

_Water Flux_
  
  $$
  J_w = K_w\bigl(\Delta P - \Delta \pi\bigr)
  $$

  $$
  J_w = \frac{Q_p}{A}
  $$


_Salt Flux_

  $$
  J_s = K_s\,C_{avg}
  $$

  $$
  J_s = C_p J_w
  $$

_Recovery_

  $$
  R = \frac{Q_p}{Q_f}
  $$

_Pump Power_ 


  $$
  W_p = \bigl(\Delta P - P_{atm} \bigr) Q_f
  $$

_Product Concentration_


  $$
  C_p <= C_{p,max}
  $$

The objective of the model is to minimize the specific energy consumption (SEC) of the system. Add the following as an `Expression` to the model:

_SEC_

  $$
  SEC = \frac{W_p}{Q_p} 
  $$

The question we are trying to answer with this simple model is: What is the membrane area required for a system operating at 100 L/s with 50% recovery?

Fix the following variables to these values:

- $K_w = 1.5 \text{ LMH/bar}$
- $K_s = 1.5 \text{ LMH}$
- $C_f = 35 \text{ g/L}$
- $Q_f = 100 \text{ L/s}$
- $R = 0.5$

If the model is set up correctly, this will result in 1 DOF (area) with $A$ = 12,000 m2 and $\text{SEC}$ = 2.8 kWh/m3.

In [ ]:
from pyomo.environ import (
    ConcreteModel,
    Var,
    Constraint,
    Expression,
    Param,
    Objective,
    SolverFactory,
    value,
    assert_optimal_termination,
    units as pyunits,
)
from idaes.core.util.model_statistics import degrees_of_freedom

solver = SolverFactory("ipopt")

In [ ]:
## Inlet Conditions

# Flow rate in L/s
initial_flow = {"feed": 100, "brine": 50, "product": 50}

# Concentration in g/L
initial_concentration = {"feed": 35, "brine": 70, "product": 1}

# Maximum product concentration in g/L
max_product_concentration = 0.5

# Names of streams for indexing Vars
streams = ["feed", "brine", "product"]

In [ ]:
# Create model

m = ConcreteModel()

## Define model parameters
m.temperature = Param(
    initialize=298.15,
    mutable=True,
    units=pyunits.kelvin,
    doc="Feed temperature",
)
m.pressure_atm = Param(
    initialize=1.0,
    mutable=True,
    units=pyunits.bar,
    doc="Atmospheric pressure",
)
m.mw_NaCl = Param(
    initialize=58.44,
    mutable=True,
    units=pyunits.gram / pyunits.mole,
    doc="Molecular weight of NaCl",
)
m.vant_hoff_factor = Param(
    initialize=0.93,
    mutable=True,
    units=pyunits.dimensionless,
    doc="Van't Hoff factor for NaCl",
)
m.osmotic_coefficient = Param(
    initialize=2,
    mutable=True,
    units=pyunits.dimensionless,
    doc="Osmotic coefficient for NaCl",
)
m.univ_gas_constant = Param(
    initialize=0.08314,
    mutable=True,
    units=(pyunits.bars * pyunits.liter) / (pyunits.mole * pyunits.kelvin),
    doc="Universal gas constant",
)
m.channel_height = Param(
    initialize=1,
    mutable=True,
    units=pyunits.mm,
    doc="Channel height of the membrane",
)
################################################################################
## Define state variables
m.flow = Var(
    streams,
    initialize=initial_flow,
    units=pyunits.liters / pyunits.s,
    bounds=(1, None),
    doc="Flow rate of different streams",
)

m.concentration_mass = Var(
    streams,
    initialize=initial_concentration,
    units=pyunits.grams / pyunits.liter,
    bounds=(0.01, 100),
    doc="Concentration of NaCl in different streams",
)


## Properties
# Molar concetration for osmotic pressure calculation
def rule_concentration_mol(m, s):
    return pyunits.convert(
        m.concentration_mass[s] / m.mw_NaCl, to_units=pyunits.mole / pyunits.liter
    )


m.concentration_mol = Expression(
    streams,
    rule=rule_concentration_mol,
)


# Osmotic pressure
m.osmotic_pressure = Var(
    streams,
    units=pyunits.bar,
    bounds=(0.001, 85),
    doc="Osmotic pressure of different streams",
)


def rule_osmotic_pressure(m, s):
    return m.osmotic_pressure[s] == pyunits.convert(
        m.osmotic_coefficient
        * m.vant_hoff_factor
        * m.concentration_mol[s]
        * m.univ_gas_constant
        * m.temperature,
        to_units=pyunits.bar,
    )


m.osmotic_pressure_constraint = Constraint(streams, rule=rule_osmotic_pressure)
# Average inlet osmotic pressure
m.osmotic_pressure_avg = Expression(
    expr=(m.osmotic_pressure["feed"] + m.osmotic_pressure["brine"]) / 2
)
# Average inlet concentration
m.concentration_mass_avg = Expression(
    expr=(m.concentration_mass["feed"] + m.concentration_mass["brine"]) / 2
)

#### Given material ends here
################################################################################
################################################################################
# ADD REST OF MODEL BELOW


# m.obj = Objective(expr=m.SEC)

# Fix variables


# 1 DOF is pressure or membrane area
print(f"dof = {degrees_of_freedom(m)}")

results = solver.solve(m, tee=False)
assert_optimal_termination(results)
print(f"termination = {results.solver.termination_condition}")

print(f"Area = {value(m.area):.1f} m2")
print(f"SEC = {value(m.SEC):.2f} kWh/m3")